# GaussianPro + Scaffold-GS — full dataset, metric 15k/3k

Notebook chạy tuần tự tất cả scene, train trên 100% ảnh COLMAP, tính photo loss, PSNR, SSIM và LPIPS từ iteration 15.000 rồi mỗi 3.000 iteration, render theo `test_poses.csv` và tạo ZIP riêng ngay sau khi mỗi scene hoàn tất.

**Lưu ý:** nhóm `val` là 20 ảnh được lấy cố định từ tập train để theo dõi xu hướng, không phải holdout độc lập. Chỉ checkpoint cuối được lưu để hạn chế dung lượng đĩa.

In [ ]:
from pathlib import Path
import csv, gc, json, os, shutil, subprocess, sys, time, zipfile
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display
from PIL import Image

# ==================== CẤU HÌNH ====================
REPO_DIR = Path('/kaggle/working/round2_kusanagi')
DATA_ROOT = Path('/kaggle/input/datasets/acomingzzz/maindataset')
OUTPUT_ROOT = Path('/kaggle/working/full_dataset_gaussianpro_30k')
SCENE_ZIP_DIR = Path('/kaggle/working/scene_zips')

# output_name -> folder trong DATA_ROOT. Sửa mapping nếu tên folder thực tế khác.
SCENES = {
    'chair': 'chair',
    'bonsai': 'bonsai',
    'HCM0421': 'HCM0421',
    'HCM0539': 'HCM0539',
    'HCM0540': 'HCM0540',
    'HCM0644': 'HCM0644',
    'HCM0674': 'HCM0674',
}

ITERATIONS = 30_000
METRIC_START = 15_000
METRIC_EVERY = 3_000
METRIC_ITERATIONS = list(range(METRIC_START, ITERATIONS + 1, METRIC_EVERY))
VALIDATION_RATIO = 0.0       # không giữ lại ảnh nào: train toàn bộ COLMAP
VALIDATION_SAMPLE_COUNT = 20 # 20 ảnh train cố định chỉ dùng để monitor val
VALIDATION_SEED = 42
TRAIN_RESOLUTION = 1
RENDER_RESOLUTION = 1
DATA_DEVICE = 'cpu'
GPU = '0'
MONITOR_SECONDS = 30
SKIP_FINISHED = True
CORRECT_RADIAL_DISTORTION = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCENE_ZIP_DIR.mkdir(parents=True, exist_ok=True)
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print('Metric iterations:', METRIC_ITERATIONS)

In [ ]:
# Chuẩn bị source và kiểm tra dataset.
if not (REPO_DIR / 'train.py').exists():
    archives = list(Path('/kaggle/input').rglob('kusanagi-source.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Cần đúng 1 kusanagi-source.zip, tìm thấy: {archives}')
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(REPO_DIR)

def resolve_train_root(scene_root: Path) -> Path:
    for root in (scene_root, scene_root / 'train'):
        if ((root / 'sparse').exists() or (root / 'transforms_train.json').exists()) and (root / 'images').exists():
            return root
    raise FileNotFoundError(f'Không tìm thấy sparse/ và images/ trong {scene_root}')

def find_pose_csv(source: Path) -> Path:
    for path in (source / 'test' / 'test_poses.csv', source.parent / 'test' / 'test_poses.csv'):
        if path.exists():
            return path
    raise FileNotFoundError(f'Không tìm thấy test_poses.csv cho {source}')

resolved = {}
for output_name, folder_name in SCENES.items():
    source = resolve_train_root(DATA_ROOT / folder_name)
    image_count = sum(p.is_file() for p in (source / 'images').iterdir())
    pose_csv = find_pose_csv(source)
    with pose_csv.open(newline='', encoding='utf-8-sig') as handle:
        test_count = sum(1 for _ in csv.DictReader(handle))
    resolved[output_name] = source
    print(f'{output_name:10s} train_images={image_count:4d} test_poses={test_count:3d} source={source}')

subprocess.run(['nvidia-smi'], check=False)
usage = shutil.disk_usage('/kaggle/working')
print(f'Free disk: {usage.free / 1024**3:.2f} GiB')

In [ ]:
# Dashboard được refresh trong lúc scene đang train.
def read_csv_safe(path):
    try:
        return pd.read_csv(path) if path.exists() else pd.DataFrame()
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return pd.DataFrame()

def plot_scene_monitor(scene_name, model_dir, final=False):
    train_df = read_csv_safe(model_dir / 'train_curve.csv')
    metrics_df = read_csv_safe(model_dir / 'validation_metrics.csv')
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'{scene_name} — train 100%, proxy-val 20 ảnh' + (' (final)' if final else ' (running)'), fontsize=15)

    ax = axes[0, 0]
    if not train_df.empty:
        ax.plot(train_df.iteration, train_df.train_total_loss.rolling(10, min_periods=1).mean(), label='train total (smooth)')
    if not metrics_df.empty:
        for split, color in [('train_eval', 'tab:green'), ('val', 'tab:red')]:
            part = metrics_df[metrics_df.split == split]
            ax.plot(part.iteration, part.photo_loss, 'o-', ms=3, color=color, label=f'{split} photo')
    ax.set_title('Train/validation loss'); ax.set_ylabel('Loss'); ax.legend()

    for ax, metric, title in [
        (axes[0, 1], 'psnr', 'PSNR ↑'),
        (axes[1, 0], 'ssim', 'SSIM ↑'),
        (axes[1, 1], 'lpips', 'LPIPS ↓'),
    ]:
        if not metrics_df.empty:
            for split, color in [('train_eval', 'tab:green'), ('val', 'tab:red')]:
                part = metrics_df[metrics_df.split == split]
                ax.plot(part.iteration, part[metric], 'o-', ms=3, color=color, label=split)
        ax.set_title(title); ax.set_ylabel(metric.upper()); ax.legend()
    for ax in axes.flat:
        ax.set_xlabel('Iteration'); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

    if not metrics_df.empty:
        display(metrics_df.sort_values(['iteration', 'split']).tail(12))

def monitor_process(process, scene_name, model_dir, log_path, started):
    while process.poll() is None:
        time.sleep(MONITOR_SECONDS)
        clear_output(wait=True)
        done = read_csv_safe(model_dir / 'validation_metrics.csv')
        last_iter = int(done.iteration.max()) if not done.empty else 0
        print(f'{scene_name}: {(time.time()-started)/60:.1f} phút | metric mới nhất={last_iter}/{ITERATIONS}')
        plot_scene_monitor(scene_name, model_dir)
    code = process.wait()
    if code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace')[-16000:] if log_path.exists() else ''
        print(tail)
        raise RuntimeError(f'Train {scene_name} thất bại, return_code={code}')

In [ ]:
GAUSSIANPRO_ARGS = [
    '--use_gaussianpro',
    '--gaussianpro_start_iter', '3000',
    '--gaussianpro_add_until_iter', '15000',
    '--gaussianpro_refine_until_iter', '24000',
    '--gaussianpro_interval', '50',
    '--gaussianpro_neighbors', '4',
    '--gaussianpro_downsample', '4',
    '--gaussianpro_patch_radius', '2',
    '--gaussianpro_patchmatch_iterations', '3',
    '--gaussianpro_min_consistent_views', '3',
    '--gaussianpro_max_photo_error', '0.25',
    '--gaussianpro_max_anchors_per_step', '128',
    '--gaussianpro_voxel_factor', '1.0',
    '--gaussianpro_max_anchor_multiplier', '1.25',
    '--lambda_gaussianpro_flatness', '0.001',
    '--lambda_gaussianpro_normal_l1', '0.001',
    '--lambda_gaussianpro_normal_cos', '0.001',
]
RADIAL_ARGS = ['--correct_radial_distortion'] if CORRECT_RADIAL_DISTORTION else []

def final_checkpoint(model_dir):
    return model_dir / 'point_cloud' / f'iteration_{ITERATIONS}' / 'point_cloud.ply'

def train_scene(scene_name, source):
    model_dir = OUTPUT_ROOT / scene_name
    model_dir.mkdir(parents=True, exist_ok=True)
    if SKIP_FINISHED and final_checkpoint(model_dir).exists() and (model_dir / 'validation_metrics.csv').exists():
        print(f'[SKIP TRAIN] {scene_name}: checkpoint và metrics đã tồn tại')
        return model_dir

    # Run không hoàn chỉnh sẽ được train lại trong thư mục scene sạch để CSV không bị trùng.
    if model_dir.exists() and any(model_dir.iterdir()):
        shutil.rmtree(model_dir)
        model_dir.mkdir(parents=True)
    log_path = model_dir / 'full_train.log'
    cmd = [
        sys.executable, 'train.py', '-s', str(source), '-m', str(model_dir),
        '-r', str(TRAIN_RESOLUTION), '--data_device', DATA_DEVICE,
        '--appearance_dim', '0', '--gpu', GPU,
        '--iterations', str(ITERATIONS),
        '--validation_ratio', str(VALIDATION_RATIO),
        '--validation_seed', str(VALIDATION_SEED),
        '--validation_sample_count', str(VALIDATION_SAMPLE_COUNT),
        '--test_iterations', *map(str, METRIC_ITERATIONS),
        '--save_iterations', str(ITERATIONS),
        '--lambda_dssim', '0.2',
    ] + GAUSSIANPRO_ARGS + RADIAL_ARGS
    print(' '.join(cmd))
    started = time.time()
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
        monitor_process(process, scene_name, model_dir, log_path, started)
    if not final_checkpoint(model_dir).exists():
        raise RuntimeError(f'{scene_name}: thiếu checkpoint cuối')
    clear_output(wait=True)
    plot_scene_monitor(scene_name, model_dir, final=True)
    return model_dir

def render_test(scene_name, source, model_dir):
    cmd = [
        sys.executable, 'render.py', '-s', str(source), '-m', str(model_dir),
        '--iteration', str(ITERATIONS), '-r', str(RENDER_RESOLUTION),
        '--data_device', DATA_DEVICE, '--eval', '--skip_train',
        '--validation_ratio', '0.0',
    ] + RADIAL_ARGS
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    render_dir = model_dir / 'test' / f'ours_{ITERATIONS}' / 'renders'
    if not render_dir.exists() or not any(render_dir.iterdir()):
        raise RuntimeError(f'{scene_name}: không có test render tại {render_dir}')
    return render_dir

In [ ]:
def save_scene_summary(scene_name, model_dir):
    metrics = pd.read_csv(model_dir / 'validation_metrics.csv')
    val = metrics[metrics.split == 'val'].sort_values('iteration')
    summary = {
        'scene': scene_name,
        'iterations': ITERATIONS,
        'best_psnr_iteration': int(val.loc[val.psnr.idxmax(), 'iteration']),
        'best_psnr': float(val.psnr.max()),
        'best_ssim_iteration': int(val.loc[val.ssim.idxmax(), 'iteration']),
        'best_ssim': float(val.ssim.max()),
        'best_lpips_iteration': int(val.loc[val.lpips.idxmin(), 'iteration']),
        'best_lpips': float(val.lpips.min()),
        'final': val.iloc[-1].to_dict(),
    }
    (model_dir / 'metric_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

def zip_scene_submission(scene_name, source, render_dir, model_dir):
    pose_csv = find_pose_csv(source)
    with pose_csv.open(newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    expected_names = [row['image_name'].strip() for row in rows]
    if not expected_names or len(expected_names) != len(set(expected_names)):
        raise RuntimeError(f'{scene_name}: image_name rỗng hoặc trùng trong CSV')

    actual = {p.name: p for p in render_dir.iterdir() if p.is_file()}
    missing, extra = sorted(set(expected_names) - set(actual)), sorted(set(actual) - set(expected_names))
    if missing or extra:
        raise RuntimeError(f'{scene_name}: missing={missing}, extra={extra}')
    for row in rows:
        path = actual[row['image_name'].strip()]
        with Image.open(path) as image:
            if row.get('width') and row.get('height') and image.size != (int(row['width']), int(row['height'])):
                raise RuntimeError(f'{path.name}: size={image.size}, expected={(row["width"], row["height"])}')
            image.verify()

    zip_path = SCENE_ZIP_DIR / f'submission_{scene_name}_{ITERATIONS}.zip'
    if zip_path.exists():
        zip_path.unlink()
    expected_entries = [(Path('submission') / scene_name / name).as_posix() for name in expected_names]
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for name, arcname in zip(expected_names, expected_entries):
            archive.write(actual[name], arcname=arcname)

    with zipfile.ZipFile(zip_path) as archive:
        bad = archive.testzip()
        names = archive.namelist()
    if bad is not None:
        raise RuntimeError(f'{scene_name}: ZIP lỗi tại {bad}')
    if names != expected_entries:
        raise RuntimeError(f'{scene_name}: tên/thứ tự ảnh trong ZIP không đúng')
    print(f'ZIP READY: {zip_path} ({zip_path.stat().st_size/1024**2:.2f} MiB)')
    return zip_path

In [ ]:
# Full pipeline: train -> metric summary -> render test -> ZIP ngay từng scene.
results = []
for scene_name, source in resolved.items():
    print(f'\n========== {scene_name} ==========')
    model_dir = train_scene(scene_name, source)
    summary = save_scene_summary(scene_name, model_dir)
    render_dir = render_test(scene_name, source, model_dir)
    zip_path = zip_scene_submission(scene_name, source, render_dir, model_dir)
    results.append({
        'scene': scene_name,
        'final_psnr': summary['final']['psnr'],
        'final_ssim': summary['final']['ssim'],
        'final_lpips': summary['final']['lpips'],
        'zip_MiB': round(zip_path.stat().st_size / 1024**2, 2),
        'zip': str(zip_path),
    })
    gc.collect()

summary_df = pd.DataFrame(results)
summary_df.to_csv(OUTPUT_ROOT / 'all_scenes_summary.csv', index=False)
display(summary_df)
print('ALL SCENES COMPLETED')

In [ ]:
# Audit toàn bộ ZIP sau khi pipeline kết thúc.
audit = []
for scene_name in resolved:
    path = SCENE_ZIP_DIR / f'submission_{scene_name}_{ITERATIONS}.zip'
    if not path.exists():
        raise FileNotFoundError(path)
    with zipfile.ZipFile(path) as archive:
        bad = archive.testzip()
        image_count = sum(name.startswith(f'submission/{scene_name}/') for name in archive.namelist())
    audit.append({'scene': scene_name, 'images': image_count, 'MiB': round(path.stat().st_size/1024**2, 2), 'valid': bad is None})
display(pd.DataFrame(audit))
print('ZIP directory:', SCENE_ZIP_DIR)